In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 59. B10 Project — Reproducible B9 research package

> B9 development pipelineを、入力hashからprediction hashまでclean processで追跡できるpackage artifactへ変換する。locked outer evaluationやproduction deploymentは行わない。

## 学習目標

- data/features/model/evaluation/registryの責務を分離できる
- numeric/TF–IDF baselineを同じfixtureから再生成できる
- config/data/code/prediction hashをrunへ固定できる
- failure injectionでhash・schema・PIT gateを検証できる
- one-command reproductionの範囲と未実装範囲を説明できる

## 前提知識

- Week 37–40のbenchmark、package、PIT、registry
- B9 development-only fixtureとouter gate

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 59


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask
assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("locked outer rows present: False")

fixture rows: 256
inner train / validation: 192 64
locked outer rows present: False


## 1. Reproduction configuration

| layer | implementation | artifact |
|---|---|---|
| data | bundled SEC-derived fixture loader | fixture + manifest SHA |
| features | training-only numeric/hashed TF–IDF | transform parameters |
| models | sparse ridge baselines | coefficients/config |
| evaluation | row + company metric | validation prediction hash |
| registry | immutable development runs | run IDs |
| report | executed Notebook/Jupyter Book | HTML |

Docker/container、DuckDB/Parquet adapter、scheduler、online servingはAdvanced deployment workで、Coreの再現packageと同一視しない。

In [4]:
import hashlib
from scipy import sparse

preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric = preprocessor.transform(fixture.numeric_features)
tfidf_model = qt.fit_hashed_tfidf(fixture.token_hashes, train_mask, maximum_features=256, minimum_document_frequency=2)
tfidf = tfidf_model.transform(fixture.token_hashes)

models = {
    "numeric-ridge": (
        qt.fit_sparse_ridge(numeric[train_mask], fixture.targets[train_mask], ridge=1.0),
        numeric[validation_mask],
        {"family": "numeric_ridge", "ridge": 1.0},
    ),
    "tfidf-ridge": (
        qt.fit_sparse_ridge(tfidf[train_mask], fixture.targets[train_mask], ridge=1.0),
        tfidf[validation_mask],
        {"family": "hashed_tfidf_ridge", "ridge": 1.0, "maximum_features": 256, "minimum_document_frequency": 2},
    ),
}
data_digest = "953c9b06c6c1dc1ef68c5e21f1ee88c4fe20d1ee34d5887150e51843184ad0b0"
registry = qt.ModelRegistry()
metric_rows = []
run_rows = []
for candidate_name, (model, validation_features, config) in models.items():
    prediction = model.predict(validation_features)
    metrics = qt.regression_error_table(fixture.targets[validation_mask], prediction, np.asarray(fixture.entity_ids)[validation_mask])
    prediction_digest = hashlib.sha256(prediction.astype("<f8").tobytes()).hexdigest()
    run = qt.build_experiment_run(
        experiment_name="b10-b9-reproduction",
        candidate_name=candidate_name,
        stage="development",
        config={**config, "outer_access": "unopened"},
        data_sha256=data_digest,
        code_revision="notebook-59-generated-source",
        metrics=metrics,
        artifact_sha256={"validation_prediction": prediction_digest},
    )
    registry = qt.register_run(registry, run)
    metric_rows.append({"model": candidate_name, **metrics})
    run_rows.append({"model": candidate_name, "run_id": run.run_id, "config_sha256": run.config_sha256, "prediction_sha256": prediction_digest})

metric_table = pd.DataFrame(metric_rows).sort_values("mae")
display(metric_table)
display(pd.DataFrame(run_rows))
assert registry.production_run_id is None

,model,mae,median_absolute_error,rmse,company_macro_mae
1,tfidf-ridge,0.061339,0.032573,0.124007,0.053581
0,numeric-ridge,0.069652,0.027980,0.156989,0.059852


,model,run_id,config_sha256,prediction_sha256
0,numeric-ridge,0953b0d917581bb205db19db,8c2f1a0137770b3e60eabb6822041bff19a4052d042c3f...,0cd72c914c8cd79c2ff190492e4130d965b9ddce54c50a...
1,tfidf-ridge,83d48fe56d41b91c7e1df992,b836117e06c932fbd86406de8b0005071321c07f35ad2d...,eb6680a2dae950ebdb3c9b756a30c0f12494cc58fce57b...


## 2. Failure injection and evidence graph

config、input、code、predictionのどれかが変わればrun evidenceも変わる。run ID一致だけでsource document integrityを再検査したことにはならないため、各layerの検証責務を残す。

In [5]:
base_run = registry.runs[0]
changed_run = qt.build_experiment_run(
    experiment_name=base_run.experiment_name,
    candidate_name=base_run.candidate_name,
    stage="development",
    config={**base_run.config, "ridge": 10.0},
    data_sha256=base_run.data_sha256,
    code_revision=base_run.code_revision,
    metrics=base_run.metrics,
    artifact_sha256=base_run.artifact_sha256,
)
assert changed_run.config_sha256 != base_run.config_sha256
assert changed_run.run_id != base_run.run_id

evidence = pd.DataFrame(
    [
        {"node": "SEC source gate", "status": "passed"},
        {"node": "teaching fixture", "status": "passed"},
        {"node": "feature transforms", "status": "passed"},
        {"node": "development runs", "status": "registered"},
        {"node": "nominee manifest", "status": "not frozen"},
        {"node": "locked outer", "status": "unopened"},
        {"node": "production", "status": "not applicable"},
    ]
)
display(evidence)

fig = go.Figure()
fig.add_bar(x=metric_table["model"], y=metric_table["mae"], name="MAE")
fig.add_bar(x=metric_table["model"], y=metric_table["company_macro_mae"], name="company macro MAE")
fig.update_layout(title="Reproduced development-only baselines", yaxis_title="Absolute log-change error", barmode="group", template="plotly_white")
fig.show()

,node,status
0,SEC source gate,passed
1,teaching fixture,passed
2,feature transforms,passed
3,development runs,registered
4,nominee manifest,not frozen
5,locked outer,unopened
6,production,not applicable


## 3. One-command contract

repository rootからの再現順序は次である。

```bash
uv sync --package quant-research-textbook
uv run --no-sync pytest analytics/quant_research/tests
uv run --no-sync python analytics/quant_research/tools/build_notebooks.py --check
uv run --no-sync jupyter-book build analytics/quant_research/book -W --keep-going --all
```

raw SEC取得はcontact-bearing User-Agentと外部cacheを必要とし、このoffline教材再現commandへ含めない。source gateを再取得する場合は別の明示手順・rate limit・manifestを使う。

## 4. 失敗モード

- `latest.pkl`をmodel registryと呼ぶ
- environmentに偶然あるdependencyを利用する
- external raw cacheなしでsource retrievalを再現したと主張する
- development baselineをproductionへpromoteする
- B9 outerをB10 engineering確認のために開く

## 5. 段階別演習

### 基礎

1. run IDを変えるinputを5種類挙げよ。
2. config変更failure injectionを再実行せよ。

### 標準

3. clean processで同じprediction hashを得よ。
4. artifact DAGをmachine-readable JSONへ変換せよ。

### 研究

5. container/remote registry追加のthreat modelとrollback drillを書け。

## 6. Exit Criteria

- [ ] package layerを分離した
- [ ] numeric/TF–IDF baselineを再生成した
- [ ] data/config/code/prediction hashをrunへ結んだ
- [ ] config mutationでrun IDが変わることをtestした
- [ ] outer/production未実装範囲を隠していない

## 7. 出典

- [Python `timeit` documentation](https://docs.python.org/3/library/timeit.html)
- [Python `multiprocessing` documentation](https://docs.python.org/3/library/multiprocessing.html)
- [NumPy CPU/SIMD optimizations](https://numpy.org/doc/stable/reference/simd/index.html)
- [IEEE 754-2019 overview](https://standards.ieee.org/ieee/754/6210/)

- [Python Packaging User Guide](https://packaging.python.org/en/latest/)
- [Python logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [pytest documentation](https://docs.pytest.org/)
- [Semantic Versioning 2.0.0](https://semver.org/)

- [Apache Arrow columnar format specification](https://arrow.apache.org/docs/format/Columnar.html)
- [Apache Parquet format](https://parquet.apache.org/docs/file-format/)
- [DuckDB documentation](https://duckdb.org/docs/stable/)
- [SQLite window functions](https://www.sqlite.org/windowfunctions.html)